# 6.5 Naive Bayes

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.05_naive-bayes.ipynb)

Companion code for Section 6.5 of the book. Run the cells in order from top to bottom.

**Example 6.5.1**

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, ComplementNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.datasets import load_breast_cancer, fetch_20newsgroups
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import numpy as np

# ── Gaussian Naive Bayes on tabular data ──────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      stratify=y, random_state=42)

gnb = GaussianNB()
gnb.fit(X_train, y_train)  # Note: GaussianNB does NOT require scaling
print('=== Gaussian Naive Bayes (Breast Cancer) ===')
print(f'Test Accuracy: {gnb.score(X_test, y_test):.4f}')
print(f'Class priors P(y):  {dict(zip(data.target_names, gnb.class_prior_.round(3)))}')
print(f'Feature means per class (first 3 features):')
for i, cls in enumerate(data.target_names):
    print(f'  {cls}: {gnb.theta_[i,:3].round(3)}')

# ── Multinomial Naive Bayes on text data ──────────────────────────
categories = ['sci.med','sci.space','comp.graphics','rec.sport.hockey']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, random_state=42)
X_text = newsgroups.data
y_text = newsgroups.target

X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(
    X_text, y_text, test_size=0.2, stratify=y_text, random_state=42)

# TF-IDF vectorisation
tfidf = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1,2))
X_tr_tfidf = tfidf.fit_transform(X_tr_t)
X_te_tfidf = tfidf.transform(X_te_t)

print('\n=== Multinomial Naive Bayes (20 Newsgroups - 4 categories) ===')
for name, model in [
    ('MultinomialNB (α=0.1)', MultinomialNB(alpha=0.1)),
    ('MultinomialNB (α=1.0)', MultinomialNB(alpha=1.0)),
    ('BernoulliNB',           BernoulliNB(alpha=0.1)),
    ('ComplementNB',          ComplementNB(alpha=0.1)),
]:
    model.fit(X_tr_tfidf, y_tr_t)
    acc = model.score(X_te_tfidf, y_te_t)
    print(f'  {name:<28}: {acc:.4f}')

# ── Naive Bayes Laplace smoothing explained ───────────────────────
print('\nLaplace smoothing (alpha parameter):')
print('  alpha=0: no smoothing - zero probability for unseen words → underflow')
print('  alpha=1: Laplace smoothing - add 1 to all counts (standard)')
print('  alpha<1: less smoothing - trusts observed frequencies more')

# Best model detailed report
best_nb = ComplementNB(alpha=0.1)
best_nb.fit(X_tr_tfidf, y_tr_t)
y_pred_nb = best_nb.predict(X_te_tfidf)
print('\nComplementNB Classification Report:')
print(classification_report(y_te_t, y_pred_nb, target_names=categories))